In [ ]:
# ===== STABLE-AC CoV (case i) — CONFIG (edit ONLY this cell) ==============

REPO_URL = "https://github.com/Avi161/ACSolverX.git"
BRANCH   = "test/stable-ac-moves-w4"   # must match the actual git branch
REPO_DIR = "ACSolverX"
CLONE       = True
UPDATE_REPO = True           # git reset --hard so a RESTART pulls latest push

# --- experiment knobs ------------------------------------------------------
# Defaults live in experiments/stable_ac/cov/config_cov.yaml (the reviewed
# file); anything set here overrides it. One resumable jsonl per budget.
CONFIG_PATH = "experiments/stable_ac/cov/config_cov.yaml"
BUDGET = [100, 1000]         # local proof budgets; production (e.g. [50000]) runs here on Colab
MODE   = "cov"               # "cov" | "baseline" (identity transform -> comparison file)
EXPERIMENT_LENGTH = False    # True = length sweep: greedy from EVERY valid CoV + a no-CoV
                             # control row per presentation (mode must be "cov");
                             # prints the length-vs-outcome digest at the end
Z_SOURCE = None              # None = yaml default ("subwords" = z from the relators).
                             # "universe" = every reduced word of len 2..universe_max_len
                             # (z need not occur — Nielsen re-coordinatisations; sweep only)
DATASETS = None              # None = the yaml's local proof pair (subset_10 + reach_1).
                             # Production 66-row benchmark:
                             # DATASETS = ["results/benchmark/subsets/benchmark_subset_60.csv",
                             #             "results/benchmark/reach/reach_tier_6.csv"]

HIGH_SPEEDUP = True          # compact fast solver (~2.9x); result-neutral — every
                             # written row is identical to a slow-mode row (solved
                             # rows are re-solved by the normal solver for the path),
                             # and files resume across the two modes. Overrides the
                             # yaml; set False to force the plain solver.

USE_CHUNKS  = False          # True = stride-split the presentations into CHUNKS
                             # chunks; each chunk writes its own _c{i}of{N}_ jsonl
CHUNKS      = 3
CHUNK_INDEX = None           # None = run ALL chunks as parallel processes in THIS
                             # session (high-RAM multi-vCPU runtime); 1..CHUNKS =
                             # run only that chunk (one per parallel Colab session)


In [ ]:
# ==================== SETUP (clone / pull / install) ======================
import os, sys, subprocess

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)                       # anchor so re-runs never nest the clone
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy pyyaml")
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    # local: walk up from cwd to the repo root (dir holding experiments/ + data/)
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

# run from repo root so relative paths and "import experiments..." resolve
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)

# a `git reset --hard` rewrites .py files but sys.modules keeps the OLD module
# objects -- drop them so RUN imports what SETUP just fetched (pull != reload)
import importlib
for _m in [m for m in sys.modules if m == "experiments" or m.startswith("experiments.")]:
    del sys.modules[_m]
importlib.invalidate_caches()


In [ ]:
# ==================== RUN =================================================
from experiments.stable_ac.cov.run_cov import run

run(config_path=CONFIG_PATH, budgets=BUDGET, mode=MODE,   # resumable; one jsonl per budget
    experiment_length=EXPERIMENT_LENGTH, z_source=Z_SOURCE, datasets=DATASETS,
    high_speedup=HIGH_SPEEDUP,
    use_chunks=USE_CHUNKS, chunks=CHUNKS, chunk_index=CHUNK_INDEX)


In [ ]:
# ==================== MERGE (chunked runs only, after ALL chunks finish) ==
# Concatenates the completed _c{i}of{N}_ chunk jsonls into the canonical
# unchunked file. It REFUSES to run while any chunk is missing rows (counts are
# re-derived by enumeration) and when the target already exists, so running it
# early or twice is safe. Verify certificates afterwards:
#   .venv/bin/python3 -m experiments.stable_ac.verify_results results/stable_ac/cov
if USE_CHUNKS:
    from experiments.stable_ac.cov.run_cov import merge_chunks
    merge_chunks(config_path=CONFIG_PATH, budgets=BUDGET, mode=MODE,
                 experiment_length=EXPERIMENT_LENGTH, z_source=Z_SOURCE,
                 datasets=DATASETS, use_chunks=True, chunks=CHUNKS)
